# Tag-Anchored Hybrid Mapper — IFRS S1/S2 (v2: statuses + evaluation)

Maps **requirements** (`evidence_tags`) → **concepts** → **payload providers**, year-aware, with
precedence, `data_gaps` join, and full provenance.

**v2 adds:** two-pass resolution (own section → whole-bank merged), a `narrative` terminal status for
procedural/umbrella clauses, and a label-free **evaluation suite** (coverage, integrity,
data/tag utilization, arithmetic/precedence).

Terminal statuses: `resolved` · `resolved_cross_section` · `declared_gap` · `narrative` · `no_bridge_entry` · `unmapped`.

In [ ]:
import json, glob, os, re
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter

REQ_DIR     = Path("gen_data/IFRS/ifrs_requirements_kb_outputs_final/section_by_section_requirements/json")
PAYLOAD_DIR = Path("gen_data/payloads_risk")
SECTION_ALIASES = {"metrics_targets": "metrics_and_targets"}
def canon_section(tok): return SECTION_ALIASES.get(tok, tok)
MANDATORY_ONLY = True   # report generation consumes mandatory requirements only
assert REQ_DIR.exists() and PAYLOAD_DIR.exists()
print("req files:", len(list(REQ_DIR.glob('*.json'))), "| payloads:", len(list(PAYLOAD_DIR.glob('payload_*.json'))))

In [ ]:
# ================= CONCEPT BRIDGE (the only file domain experts edit) =================
CANON, EVID = "canonical", "evidence"
BRIDGE = {
 "ghg_emissions":   [("reporting_kpis",["scope1_2024_tco2e","scope2_market_2024_tco2e","scope2_location_2024_tco2e","scope3_travel_2024_tco2e"],CANON),
                     ("scope1",None,EVID),("scope2",None,EVID),("scope3_travel",None,EVID),
                     ("scope3_categories",["category_number","category_name","emissions_tco2e","included_flag"],EVID),
                     ("ghg_methodology",None,EVID),("scope12_consolidation",None,EVID)],
 "scope_1":[("reporting_kpis",["scope1_2024_tco2e"],CANON),("scope1",None,EVID),("scope12_consolidation",None,EVID)],
 "scope_2":[("reporting_kpis",["scope2_market_2024_tco2e","scope2_location_2024_tco2e"],CANON),("scope2",None,EVID)],
 "scope_3":[("reporting_kpis",["scope3_travel_2024_tco2e"],CANON),("scope3_travel",None,EVID),
            ("scope3_categories",None,EVID),("financed_emissions_equity",None,EVID),("financed_emissions_sovereign",None,EVID)],
 "financed_emissions":[("reporting_kpis",["financed_emissions_2024_tco2e","carbon_intensity_2024_tco2e_per_meur"],CANON),
                       ("financed_emissions",None,EVID),("financed_emissions_equity",None,EVID),("financed_emissions_sovereign",None,EVID)],
 "targets":[("reporting_kpis",["target_summary"],CANON),("targets",None,EVID)],
 "carbon_credits":[("reporting_kpis",["carbon_credit_summary"],CANON),("carbon_credits",None,EVID)],
 "metrics":[("reporting_kpis",None,CANON),("financial_summary",None,EVID),("scope1",None,EVID),
            ("scope2",None,EVID),("scope3_categories",None,EVID),("financed_emissions",None,EVID),("internal_carbon_price",None,EVID)],
 "materiality":[("climate_risk_register",None,EVID),
                ("value_chain_map",["node_name","materiality_flag","sustainability_theme","climate_exposure_type"],EVID),
                ("climate_financial_effects",None,EVID),("general_requirements_context",None,EVID)],
 "governance_body":[("reporting_kpis",["governance_maturity"],CANON),("governance",None,EVID),("board_minutes",None,EVID)],
 "management_role":[("governance",None,EVID),("board_minutes",None,EVID)],
 "remuneration":[("reporting_kpis",["governance_maturity"],CANON),
                 ("governance",["ceo_esg_compensation_pct","ceo_compensation_esg_linked","all_exec_climate_remuneration_pct"],EVID)],
 "risk_process":[("climate_risk_register",None,EVID)],
 "scenario_analysis":[("climate_scenarios",None,EVID),("resilience_assessment",None,EVID)],
 "financial_effects":[("climate_financial_effects",None,EVID),("financial_summary",None,EVID)],
 "business_model_value_chain":[("value_chain_map",None,EVID),("transition_plan",None,EVID),("climate_opportunities",None,EVID)],
 "strategy_decision_making":[("transition_plan",None,EVID),("resilience_assessment",None,EVID),("climate_opportunities",None,EVID)],
 "connected_information":[("financial_summary",None,EVID),("climate_financial_effects",None,EVID),("general_requirements_context",None,EVID)],
 "commercial_banking":[("financed_emissions",None,EVID),("financial_summary",None,EVID)],
 "asset_management":[("financed_emissions_equity",None,EVID)],
 "source_guidance":[("ghg_methodology",None,EVID),("general_requirements_context",None,EVID)],
 # --- v2 additions (data-backed; verified against the residuals) ---
 "measurement_uncertainty":[("financed_emissions_equity",["pcaf_data_quality_score","proxy_confidence","emissions_proxy_used","proxy_basis","proxy_reason"],EVID),
                            ("financed_emissions_sovereign",["pcaf_data_quality_score","proxy_confidence","emissions_proxy_used","proxy_basis","proxy_reason"],EVID),
                            ("ghg_methodology",None,EVID),("general_requirements_context",None,EVID)],
 "insurance":[("financed_emissions_equity",["nace_code","asset_class","esg_classification"],EVID),   # stopgap; durable fix = re-tag B63A upstream
              ("financed_emissions_sovereign",["nace_code","country"],EVID),("general_requirements_context",None,EVID)],
}
PREC = {"metadata_correction": -1, CANON: 0, EVID: 1}
def unit_of(f):
    for suf,u in [("_tco2e","tCO2e"),("_meur","MEUR"),("_pct","%"),("_eur","EUR"),("_c","degC")]:
        if f.endswith(suf): return u
    return None
def id_field_of(rec):
    for k in rec:
        if k.endswith("_id") and k!="bank_id": return k
    return "bank_id"
print("concepts in bridge:", len(BRIDGE))

In [ ]:
def load_requirements(section_key):
    for f in REQ_DIR.glob("*.json"):
        try: d=json.loads(f.read_text(encoding="utf-8"))
        except Exception: continue
        if d.get("section_key")==section_key:
            reqs=[r for _,b in d.get("standards",{}).items() for r in b.get("requirements",[])]
            return [r for r in reqs if r.get("mandatory")] if MANDATORY_ONLY else reqs
    return []

def load_payloads():
    P={}
    for f in PAYLOAD_DIR.glob("payload_*.json"):
        bank,tok=f.stem[len("payload_"):].split("_",1)
        P.setdefault(bank,{})[canon_section(tok)]=json.loads(f.read_text(encoding="utf-8"))
    return P

def merge_bank_payload(bank, P):
    """Union of all section payloads for a bank: collections deduped by record id, data_gaps unioned.
    Used for the cross-section 2nd pass (umbrella clauses are satisfied by sibling sections)."""
    merged={"metadata":{"data_gaps":[]}}
    year=None
    for sec,pl in P[bank].items():
        year=year or pl.get("metadata",{}).get("reporting_year")
        merged["metadata"]["data_gaps"] += pl.get("metadata",{}).get("data_gaps",[])
        for k,v in pl.items():
            if k in ("metadata",): continue
            if isinstance(v,list):
                dst=merged.setdefault(k,[]); have={str(r.get(id_field_of(r))) for r in dst if isinstance(r,dict)}
                for r in v:
                    if isinstance(r,dict) and str(r.get(id_field_of(r))) not in have: dst.append(r)
            elif isinstance(v,dict):
                merged.setdefault(k,{}).update(v)
    merged["metadata"]["reporting_year"]=year
    return merged

PAYLOADS=load_payloads(); BANKS=sorted(PAYLOADS)
MERGED={b:merge_bank_payload(b,PAYLOADS) for b in BANKS}
print("banks:",BANKS)

In [ ]:
USE_EMBEDDINGS=False
CONCEPT_LEX={c:set(re.findall(r"[a-z0-9]+",c)) for c in BRIDGE}
def infer_concepts_lexical(text):
    toks=set(re.findall(r"[a-z0-9]+",(text or "").lower()))
    sc=[(len(l&toks)/max(len(l),1),c) for c,l in CONCEPT_LEX.items() if l&toks]
    sc.sort(reverse=True); return [(c,min(0.6,s)) for s,c in sc[:2]]
def infer_concepts_embedding(text): raise NotImplementedError("wire BGE-M3 / MPNet here")
def infer_concepts(text): return infer_concepts_embedding(text) if USE_EMBEDDINGS else infer_concepts_lexical(text)

# procedural / umbrella clauses -> satisfied by a boilerplate statement, not by data
NARRATIVE_PATTERNS=[r"shall apply paragraph", r"refer to and consider", r"consider the applicability",
                    r"provide disclosures about", r"in accordance with (?:the )?paragraph"]
def is_narrative(req):
    t=(req.get("requirement_text") or "").lower()
    return any(re.search(p,t) for p in NARRATIVE_PATTERNS)

In [ ]:
def select_records(payload, collection, year, include_comparatives=False):
    if collection not in payload: return []
    v=payload[collection]
    if isinstance(v,dict): return [v]
    if not isinstance(v,list): return []
    out=[]
    for r in v:
        if not isinstance(r,dict): continue
        if "reporting_year" in r and not include_comparatives and r["reporting_year"]!=year: continue
        out.append(r)
    return out

def match_gaps(concepts, data_gaps, year):
    pf,pc=set(),set()
    for c in concepts:
        for coll,fields,_ in BRIDGE.get(c,[]):
            pc.add(coll)
            if fields: pf|=set(fields)
    hits=[]
    for g in data_gaps or []:
        fld=g.get("field",""); base=fld.split(".")[0]
        if fld in pf or base in pc or base in pf or any(base in f for f in pf):
            if (not g.get("affected_years")) or (year in g["affected_years"]): hits.append(g)
    return hits

In [ ]:
@dataclass
class Evidence:
    requirement_id:str; concept:str; collection:str; record_id:str
    field:str; value:object; unit:str; reporting_year:object
    precedence:int; layer:str; is_primary:bool=False

def resolve(req, payload, year):
    tags=req.get("evidence_tags") or []
    layer,concepts=("L1",[(t,1.0) for t in tags]) if tags else ("L3_lexical",infer_concepts(req.get("requirement_text","")))
    slices=[]
    for c,_ in concepts:
        for coll,fields,pk in BRIDGE.get(c,[]):
            for rec in select_records(payload,coll,year):
                use=fields if fields else [k for k in rec if k!="bank_id"]
                rid=rec.get(id_field_of(rec),rec.get("bank_id")); ry=rec.get("reporting_year",year)
                for f in use:
                    val=rec.get(f)
                    if val is None or (isinstance(val,(dict,list)) and not val): continue
                    slices.append(Evidence(req["requirement_id"],c,coll,str(rid),f,val,unit_of(f),ry,PREC[pk],layer))
    byc=defaultdict(list)
    for s in slices: byc[s.concept].append(s)
    for c,ss in byc.items():
        m=min(x.precedence for x in ss)
        for x in ss:
            if x.precedence==m: x.is_primary=True
    gaps=match_gaps([c for c,_ in concepts],payload.get("metadata",{}).get("data_gaps",[]),year)
    nop=[c for c,_ in concepts if c not in BRIDGE]
    status="resolved" if slices else ("declared_gap" if gaps else ("no_bridge_entry" if nop and len(nop)==len(concepts) else "unmapped"))
    conf=round((1.0 if tags else max([s for _,s in concepts] or [0.0]))*req.get("requirement_quality_score",1.0),3)
    return dict(requirement_id=req["requirement_id"],standard=req.get("standard"),mandatory=req.get("mandatory"),
                banking_relevance=req.get("banking_relevance"),tags=tags,layer=layer,status=status,confidence=conf,
                cross_section=False,n_evidence=len(slices),gap_fields=[g.get("field") for g in gaps],
                concepts_no_provider=nop,evidence=[asdict(s) for s in slices])

def resolve_final(req, payload, merged, year):
    """Two-pass: own section -> whole-bank merged (flagged, down-weighted) -> narrative -> residual."""
    m=resolve(req,payload,year)
    if m["status"] in ("resolved","declared_gap"): return m
    m2=resolve(req,merged,year)
    if m2["status"]=="resolved":
        m2["status"]="resolved_cross_section"; m2["cross_section"]=True
        m2["confidence"]=round(m2["confidence"]*0.7,3)
        return m2
    if is_narrative(req):
        m["status"]="narrative"; m["confidence"]=0.0
    return m

def run_bank_section(bank, section_key):
    pl=PAYLOADS[bank][section_key]; year=pl.get("metadata",{}).get("reporting_year")
    return [resolve_final(r,pl,MERGED[bank],year) for r in load_requirements(section_key)], year

In [ ]:
BANK=BANKS[0]
all_maps=[]; print(f"=== {BANK} ===")
for sec in sorted(PAYLOADS[BANK]):
    maps,year=run_bank_section(BANK,sec); all_maps+=maps
    c=Counter(m["status"] for m in maps)
    print(f"{sec:22s} y={year} reqs={len(maps):3d}  "+"  ".join(f"{k}={v}" for k,v in c.most_common()))
print("\nOVERALL:",dict(Counter(m['status'] for m in all_maps)),"TOTAL",len(all_maps))

In [ ]:
# ---------- COVERAGE ----------
COVERED={"resolved","resolved_cross_section","declared_gap","narrative"}
REL={"high":0,"medium":1,"low":2}
open_reqs=[m for m in all_maps if m['status'] not in COVERED]
open_reqs.sort(key=lambda m:(not m['mandatory'],REL.get(m['banking_relevance'],3)))
print(f"OPEN (unmapped/no_bridge_entry): {len(open_reqs)}  mandatory={sum(m['mandatory'] for m in open_reqs)}")
for m in open_reqs[:15]:
    print(f"  {m['requirement_id']:16s} mand={str(m['mandatory']):5s} rel={m['banking_relevance']:7s} tags={m['tags']} {m['status']}")
miss=Counter(c for m in all_maps for c in m['concepts_no_provider'])
print("\nMISSING BRIDGE ENTRIES:",dict(miss))
print("\nUNUSED COLLECTIONS per section:")
for sec in sorted(PAYLOADS[BANK]):
    maps,_=run_bank_section(BANK,sec)
    used={e['collection'] for m in maps for e in m['evidence']}
    present={k for k,v in PAYLOADS[BANK][sec].items() if k not in ('bank','metadata') and isinstance(v,(list,dict)) and v}
    u=sorted(present-used)
    if u: print(f"  {sec:22s}: {u}")

In [ ]:
# ---------- EXPORT: reference mode (UTF-8, traceable) ----------
NOISE={"bank_id","reporting_year","is_synthetic","data_source","currency"}
def disclosure_fields(rec):
    idf=id_field_of(rec)
    return [k for k,v in rec.items() if k not in NOISE and k!=idf and v is not None and not (isinstance(v,(dict,list)) and not v)]

def collapse(m):
    agg=defaultdict(lambda:{"record_ids":[],"precedence":9})
    for e in m["evidence"]:
        a=agg[(e["concept"],e["collection"])]
        if e["record_id"] not in a["record_ids"]: a["record_ids"].append(e["record_id"])
        a["precedence"]=min(a["precedence"],e["precedence"])
    refs=[dict(concept=c,collection=coll,precedence=v["precedence"],record_ids=v["record_ids"]) for (c,coll),v in agg.items()]
    for c in {r["concept"] for r in refs}:
        cs=[r for r in refs if r["concept"]==c]; mn=min(r["precedence"] for r in cs)
        for r in cs: r["is_primary"]=(r["precedence"]==mn)
    return refs

OUT=Path("mapping_outputs"); OUT.mkdir(exist_ok=True)
for bank in BANKS:
    store=defaultdict(dict); rows=[]
    merged=MERGED[bank]
    for sec in sorted(PAYLOADS[bank]):
        pl=PAYLOADS[bank][sec]; year=pl.get("metadata",{}).get("reporting_year")
        for r in load_requirements(sec):
            m=resolve_final(r,pl,merged,year); refs=collapse(m)
            for ref in refs:
                for rid in ref["record_ids"]:
                    rec=next((x for x in (merged.get(ref["collection"]) if isinstance(merged.get(ref["collection"]),list) else [merged.get(ref["collection"],{})])
                              if isinstance(x,dict) and str(x.get(id_field_of(x),x.get("bank_id")))==rid), None)
                    if rec is not None: store[ref["collection"]][rid]=rec
            rows.append(dict(section=sec,requirement_id=m["requirement_id"],standard=m["standard"],mandatory=m["mandatory"],
                             banking_relevance=m["banking_relevance"],tags=m["tags"],layer=m["layer"],status=m["status"],
                             confidence=m["confidence"],cross_section=m["cross_section"],
                             n_records=sum(len(r_["record_ids"]) for r_ in refs),gap_fields=m["gap_fields"],
                             concepts_no_provider=m["concepts_no_provider"],evidence_refs=refs))
    (OUT/f"mapping_{bank}.json").write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding="utf-8")
    (OUT/f"evidence_store_{bank}.json").write_text(json.dumps({k:dict(v) for k,v in store.items()},ensure_ascii=False,indent=2),encoding="utf-8")
    print(f"{bank}: mapping {(OUT/f'mapping_{bank}.json').stat().st_size/1e6:.2f}MB + store {(OUT/f'evidence_store_{bank}.json').stat().st_size/1e6:.2f}MB ({len(rows)} reqs)")

## Evaluation suite (label-free)

All checks below need **no ground truth** and run as the CI gate on every bank. Relevance
(is the linked evidence the *right* evidence) and confidence calibration are NOT computed here —
they require a small human-labeled gold sample, since no trustworthy automatic truth signal exists
in the payloads.

In [ ]:
# ---------- EVAL ----------
def evaluate(bank):
    rows=json.loads((OUT/f"mapping_{bank}.json").read_text(encoding="utf-8"))
    store=json.loads((OUT/f"evidence_store_{bank}.json").read_text(encoding="utf-8"))
    reqs={r["requirement_id"]:r for sec in PAYLOADS[bank] for r in load_requirements(sec)}
    R={}

    # 1. COVERAGE
    st=Counter(r["status"] for r in rows); n=len(rows)
    mand=[r for r in rows if r["mandatory"]]
    COVERED={"resolved","resolved_cross_section","declared_gap","narrative"}
    R["coverage_rate"]=round(sum(st[k] for k in ("resolved","resolved_cross_section"))/n,3)
    R["mandatory_disposition_rate"]=round(sum(1 for r in mand if r["status"] in COVERED)/len(mand),3)
    R["status_mix"]=dict(st)

    # 2. INTEGRITY
    dangling=0; refd=set(); stored=set((c,rid) for c,v in store.items() for rid in v)
    for r in rows:
        for ref in r["evidence_refs"]:
            for rid in ref["record_ids"]:
                if ref["collection"] not in store or rid not in store[ref["collection"]]: dangling+=1
                else: refd.add((ref["collection"],rid))
    R["referential_integrity_ok"]=(dangling==0)
    R["data_utilization"]=round(len(refd)/max(len(stored),1),3)   # records actually cited

    # 3. TAG UTILIZATION (config debt)
    all_tags=set()
    for rq in reqs.values(): all_tags|=set(rq.get("evidence_tags") or [])
    R["tag_utilization"]=round(sum(1 for t in all_tags if t in BRIDGE)/max(len(all_tags),1),3)

    # 4. ARITHMETIC / PRECEDENCE (deterministic checks pass-rate)
    checks=[]
    for sec in PAYLOADS[bank]:
        pl=PAYLOADS[bank][sec]; y=pl.get("metadata",{}).get("reporting_year")
        for r in pl.get("scope1",[]):
            if r.get("reporting_year")==y:
                checks.append(abs(r.get("scope1_gas_tco2e",0)+r.get("scope1_fleet_tco2e",0)-r.get("scope1_total_tco2e",0))<1e-4)
        for i in pl.get("financed_emissions_equity",[])+pl.get("financed_emissions_sovereign",[]):
            checks.append(0<=i.get("attribution_factor",0)<=1)
    R["arithmetic_pass_rate"]=round(sum(checks)/max(len(checks),1),3)

    return R

import pprint
pprint.pp(evaluate(BANKS[0]))

## Generation blocks — the writer's input contract

Turns the mapping into section-generation units. One **disclosure block** = one `(section, concept)`
holding the aligned S1+S2 requirements plus a bounded, precedence-resolved evidence bundle. Each
evidence item carries a **JSON Pointer** into the evidence store (`/collection/record_id/field`) and a
stable `citation_id` the writer emits inline and the deterministic gate re-resolves. Multi-standard
concepts stay whole (S1+S2 together); only oversized single-standard concepts paginate.

In [ ]:
# ---------- build generation blocks (reads mapping_outputs, writes generation_blocks_<bank>.json) ----------
from collections import OrderedDict
from itertools import groupby

SPLIT_THRESHOLD = 20   # paginate single-standard concept groups larger than this
EVIDENCE_CAP    = 40   # inline evidence per block; the rest stay reachable via the store + pointers

# concept specificity: lower = more specific = preferred as the block lead (splits scopes out of ghg_emissions)
SPEC = {c: 0 for c in ["scope_1","scope_2","scope_3","financed_emissions","carbon_credits","remuneration",
        "measurement_uncertainty","insurance","asset_management","governance_body","management_role",
        "scenario_analysis","commercial_banking"]}
for c in ["ghg_emissions","targets","risk_process","financial_effects","business_model_value_chain",
          "strategy_decision_making","materiality","source_guidance"]: SPEC[c] = 1
for c in ["metrics","connected_information"]: SPEC[c] = 2

def _lead_concept(m):
    refs = m.get("evidence_refs") or []
    if refs:
        pool = [r for r in refs if r.get("is_primary")] or refs
        return min(pool, key=lambda r: (SPEC.get(r["concept"], 1), r["precedence"], r["concept"]))["concept"]
    tags = m.get("tags") or []
    return sorted(tags, key=lambda t: (SPEC.get(t, 1), t))[0] if tags else "_general"

def _json_pointer(*parts):                                  # RFC 6901
    esc = lambda s: str(s).replace("~", "~0").replace("/", "~1")
    return "/" + "/".join(esc(p) for p in parts)

def _paginate(reqs, reqidx, thr):
    key = lambda r: (r["standard"], str(reqidx.get(r["requirement_id"], {}).get("paragraph_id")))
    rs = sorted(reqs, key=lambda r: (key(r), str(reqidx.get(r["requirement_id"], {}).get("clause_path"))))
    parts, cur = [], []
    for _, g in groupby(rs, key=key):                       # keep a paragraph's clauses in one part
        g = list(g)
        if cur and len(cur) + len(g) > thr: parts.append(cur); cur = []
        cur += g
        while len(cur) > thr: parts.append(cur[:thr]); cur = cur[thr:]
    if cur: parts.append(cur)
    return parts

def build_generation_blocks(bank):
    mapping = json.loads((OUT / f"mapping_{bank}.json").read_text(encoding="utf-8"))
    store   = json.loads((OUT / f"evidence_store_{bank}.json").read_text(encoding="utf-8"))
    reqidx  = {r["requirement_id"]: r for sec in PAYLOADS[bank] for r in load_requirements(sec)}
    gaps    = {g.get("field"): g for g in MERGED[bank].get("metadata", {}).get("data_gaps", [])}

    cit, seq = {}, [0]
    def cid(path):
        if path not in cit: seq[0] += 1; cit[path] = f"E-{bank}-{seq[0]:04d}"
        return cit[path]

    def _pinned_fields(concept, collection):
        for coll, fields, _ in BRIDGE.get(concept, []):
            if coll == collection:
                return fields            # None => all disclosure fields
        return None

    def block_evidence(part, block_concept):
        ev = OrderedDict()
        for r in part:
            for ref in r.get("evidence_refs") or []:
                coll = ref["collection"]
                pinned = _pinned_fields(ref["concept"], coll)   # respect the concept's field selection (no reporting_kpis bleed)
                for rid in ref["record_ids"]:
                    rec = store.get(coll, {}).get(rid)
                    if rec is None: continue
                    fields = [f for f in pinned if f in rec] if pinned else disclosure_fields(rec)
                    for field in fields:
                        if rec.get(field) is None or (isinstance(rec.get(field),(dict,list)) and not rec.get(field)): continue
                        path = _json_pointer(coll, rid, field)
                        if path not in ev:
                            ev[path] = {"citation_id": cid(path), "path": path, "value": rec[field],
                                        "unit": unit_of(field), "reporting_year": rec.get("reporting_year"),
                                        "collection": coll, "record_id": rid, "field": field,
                                        "concept": ref["concept"],
                                        "precedence": "canonical" if ref["precedence"] == 0 else "evidence",
                                        "is_primary": bool(ref.get("is_primary"))}
                        else:
                            if ref["precedence"] == 0: ev[path]["precedence"] = "canonical"
                            ev[path]["is_primary"] |= bool(ref.get("is_primary"))
        ev = list(ev.values())
        ev.sort(key=lambda e: (0 if e["concept"] == block_concept else 1,      # block's own concept first
                               0 if e["precedence"] == "canonical" else 1,
                               0 if e["is_primary"] else 1))
        return ev

    groups = defaultdict(list)
    for m in mapping: groups[(m["section"], _lead_concept(m))].append(m)

    blocks = []
    for (section, concept), reqs in groups.items():
        stds = sorted({r["standard"] for r in reqs})
        ids_by_std = defaultdict(list)
        for r in reqs: ids_by_std[r["standard"]].append(r["requirement_id"])
        parts = _paginate(reqs, reqidx, SPLIT_THRESHOLD) if (len(stds) == 1 and len(reqs) > SPLIT_THRESHOLD) else [reqs]
        for pi, part in enumerate(parts, 1):
            st = {r["status"] for r in part}
            mode = ("data_backed" if any(s in ("resolved","resolved_cross_section") for s in st)
                    else "absence" if "declared_gap" in st else "narrative" if "narrative" in st else "unmapped")
            req_out = []
            for r in sorted(part, key=lambda r: (r["standard"], str(reqidx.get(r["requirement_id"],{}).get("paragraph_id")),
                                                 str(reqidx.get(r["requirement_id"],{}).get("clause_path")))):
                rid = r["requirement_id"]; kb = reqidx.get(rid, {})
                other = [x for s2, lst in ids_by_std.items() if s2 != r["standard"] for x in lst]
                xref = ["IFRS S1 \u00b6" + mm.group(1)
                        for mm in re.finditer(r"IFRS S1[^.;]{0,40}?paragraph[s]?\s*([0-9]+)", kb.get("requirement_text",""))] \
                       if r["standard"] == "IFRS S2" else []
                req_out.append({"requirement_id": rid, "standard": r["standard"], "paragraph_id": kb.get("paragraph_id"),
                                "clause_path": kb.get("clause_path"), "obligation_type": kb.get("obligation_type"),
                                "mandatory": r["mandatory"], "status": r["status"], "aligned_with": other,
                                "cross_references": xref, "requirement_text": kb.get("requirement_text","")})
            ev = block_evidence(part, concept)
            _canon = [e for e in ev if e["precedence"] == "canonical"]    # curated headline numbers: never truncated
            _rest  = [e for e in ev if e["precedence"] != "canonical"]    # ordered concept-first, primary-first
            ev_kept = _canon + _rest[:max(0, EVIDENCE_CAP - len(_canon))]
            gf = sorted({g for r in part for g in (r.get("gap_fields") or [])})
            dg = [{"field": x, **{k: gaps[x][k] for k in ("affected_years","reason","instruction") if x in gaps and k in gaps[x]}} for x in gf]
            bid = f"{section}::{concept}" + (f"#{pi:02d}" if len(parts) > 1 else "")
            blocks.append({"block_id": bid, "section_key": section, "concept": concept, "bank_id": bank,
                           "part": pi, "part_count": len(parts), "generation_mode": mode, "standards_present": stds,
                           "requirements": req_out, "evidence": ev_kept,
                           "evidence_truncated": len(ev) - len(ev_kept), "data_gaps": dg})
    return blocks, store

def _deref(store, path):
    cur = store
    for p in path.strip("/").split("/"): cur = cur[p.replace("~1","/").replace("~0","~")]
    return cur

for bank in BANKS:
    blocks, store = build_generation_blocks(bank)
    (OUT / f"generation_blocks_{bank}.json").write_text(json.dumps(blocks, ensure_ascii=False, indent=2), encoding="utf-8")
    placed = [r["requirement_id"] for b in blocks for r in b["requirements"]]
    bad = sum(1 for b in blocks for e in b["evidence"] if _deref(store, e["path"]) != e["value"])
    noprim = sum(1 for b in blocks if b["generation_mode"]=="data_backed" and not any(e["is_primary"] for e in b["evidence"]))
    print(f"{bank}: {len(blocks)} blocks | reqs placed {len(placed)} (dup {len(placed)-len(set(placed))}) | "
          f"pointer mismatches {bad} | data_backed w/o primary {noprim} | modes {dict(Counter(b['generation_mode'] for b in blocks))}")
